# Verifier-as-Reward fine-tuning (Qwen2.5-0.5B, T4)

Runs the paper's training experiment: REINFORCE with the deterministic
authorization verifier as the only reward source.

**Before running:** Runtime → Change runtime type → **T4 GPU** → Save.
Then Runtime → Run all (~2–2.5 h total).

Two arms × 3 seeds × 400 steps:
- **Arm A (naive)**: symmetric ±1 reward — documents the always-refuse collapse.
- **Arm B (mitigated)**: `--entropy-beta 0.01 --balance-reward` — the learning run.

The last cell downloads all six `training_log_*.jsonl` files.

In [ ]:
!git clone https://github.com/esmaeil-abedi-dev/verifier-as-reward.git
%cd verifier-as-reward
!nvidia-smi -L
import torch
print("torch", torch.__version__, "cuda:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU"

In [ ]:
# Guardrail: the verifier (the reward source) must be intact on this box.
!PYTHONPATH=. python test_authority_verifier.py

In [ ]:
# Arm A — naive baseline, 3 seeds (expected: always-refuse collapse;
# violation rate -> 0 while false-refuse -> 1; that is a reported result)
!for s in 7 8 9; do \
  PYTHONPATH=. python train_verifier_reward.py --model Qwen/Qwen2.5-0.5B \
    --steps 400 --batch-size 8 --lr 1e-5 --eval-every 20 --eval-max-actions 80 \
    --seed $s --log-file training_log_qwen05b_naive_seed$s.jsonl \
    --save-dir ckpt_naive_seed$s ; done

In [ ]:
# Arm B — mitigated: entropy bonus + class-balanced reward, 3 seeds
!for s in 7 8 9; do \
  PYTHONPATH=. python train_verifier_reward.py --model Qwen/Qwen2.5-0.5B \
    --steps 400 --batch-size 8 --lr 1e-5 --entropy-beta 0.01 --balance-reward \
    --eval-every 20 --eval-max-actions 80 \
    --seed $s --log-file training_log_qwen05b_mitigated_seed$s.jsonl \
    --save-dir ckpt_mitigated_seed$s ; done

In [ ]:
# Ladder row: score the trained (mitigated, seed 7) checkpoint on the SAME
# natural-language prompts + parsing as the API-model proof-of-life ladder.
!PYTHONPATH=. python train_verifier_reward.py \
  --eval-checkpoint ckpt_mitigated_seed7 --test-file benchmark_test.jsonl
!PYTHONPATH=. python train_verifier_reward.py \
  --eval-checkpoint ckpt_naive_seed7 --test-file benchmark_test.jsonl

In [ ]:
# Package and download every training log (small; checkpoints stay here —
# copy ckpt_* to Drive manually if you want to keep the weights).
!zip -j training_logs.zip training_log_qwen05b_*.jsonl
from google.colab import files
files.download("training_logs.zip")